# Machyon: download a list of images

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DivyanshuSingh96/machyon/blob/main/notebooks/machyon-colab.ipynb)

Paste image links or upload a CSV, run the cells from top to bottom, and get the images back as a ZIP or in your
Google Drive. [Machyon](https://divyanshusingh96.github.io/machyon/) tunes its own speed while it downloads,
retries what can recover, keeps only real images, and tells you why anything it couldn't get failed.

Colab runs this on a machine of its own, and its files are deleted when the session ends: the last step saves the
images as a ZIP or to your Drive. Want an app on your own computer instead? `pip install machyon`, then `machyon ui`.

**How to use it:** run each cell with ▶ (or *Runtime → Run all*). Cells marked with a ⚙ have a form on the
right: change a value there, then run the cell again. If a site refuses the images, the report says what to
change. It also runs in Jupyter, where uploading and Google Drive
are replaced by a file path and a local ZIP.

In [ ]:
#@title Install Machyon
%pip install -q "machyon>=1.2.1"

In [ ]:
#@title Start
import re
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # Jupyter without ipywidgets: a text progress bar
import machyon  # noqa: E402

REQUIRED = (1, 2, 1)
if tuple(int(n) for n in re.findall(r"\d+", machyon.__version__)[:3]) < REQUIRED:
    raise RuntimeError(f"Machyon {machyon.__version__} is loaded, but this notebook needs 1.2.1 or newer. Run the install "
                       "cell again, then Runtime → Restart session, and run from the top.")

IN_COLAB = "google.colab" in sys.modules
WORK = Path.cwd()
print(f"Machyon {machyon.__version__} in {'Google Colab' if IN_COLAB else 'Jupyter'}; files go under {WORK}")

## 1. Choose your images

Pick where the links come from, the column that holds them (leave it empty and Machyon finds it), and how the files
are named. Sources can be CSV, TSV, TXT (one URL per line), JSON, JSONL or Parquet.

In [ ]:
#@title ⚙ Settings
#@markdown **Where the links come from**
SOURCE = "paste below"  #@param ["paste below", "upload a file", "file path"]
#@markdown Only for *file path*: a file in this runtime, e.g. `/content/drive/MyDrive/urls.csv`
FILE_PATH = ""  #@param {type:"string"}
#@markdown Column with the image links. Empty: Machyon guesses it (`url`, `image_url`, `image_link`, ...).
URL_COLUMN = ""  #@param {type:"string"}
#@markdown ---
#@markdown **Filenames.** Empty: `{hash}{ext}`, a stable name made from the URL. Keys: `index`, `hash`, `stem`,
#@markdown `ext`, `host` and any column of your file, e.g. `{category}/{name}{ext}` (a `/` makes a folder).
FILENAME_TEMPLATE = ""  #@param {type:"string"}
OUTPUT_DIR = "images"  #@param {type:"string"}
#@markdown ---
#@markdown **Speed.** Adaptive finds the fastest concurrency the server tolerates, starting from *Concurrency*.
ADAPTIVE = True  #@param {type:"boolean"}
CONCURRENCY = 32  #@param {type:"integer"}
MAX_CONCURRENCY = 256  #@param {type:"integer"}
MAX_RETRIES = 5  #@param {type:"integer"}
#@markdown ---
#@markdown **If a site refuses the images** (HTTP 403, or web pages instead of images): many sites only serve their
#@markdown images to their own pages. Put the address of a page the images appear on here, e.g. `https://example.com/`.
REFERER = ""  #@param {type:"string"}
#@markdown Seconds to wait for a slow server before retrying.
TIMEOUT = 15  #@param {type:"number"}

OPTIONS = dict(adaptive=ADAPTIVE, concurrency=CONCURRENCY, max_concurrency=max(MAX_CONCURRENCY, CONCURRENCY),
               max_retries=MAX_RETRIES, timeout=float(TIMEOUT))
if REFERER.strip():
    OPTIONS["referer"] = REFERER.strip()
print("Settings saved.")

### Paste links (when the source is *paste below*)

One URL per line; blank lines and lines starting with `#` are ignored. The examples are photos from
[Unsplash](https://unsplash.com/license), served by [Lorem Picsum](https://picsum.photos/); the
[Cookbook manifest](https://github.com/DivyanshuSingh96/machyon/blob/main/examples/cookbook/first_dataset/manifest.csv)
credits each photographer. The last link is broken on purpose, so you can see how a failure is reported.

In [ ]:
URLS = """
https://picsum.photos/id/40/640/427.jpg
https://picsum.photos/id/169/640/427.jpg
https://picsum.photos/id/200/640/427.jpg
https://picsum.photos/id/10/640/427.jpg
https://picsum.photos/id/15/640/427.jpg
https://picsum.photos/id/29/640/427.jpg
https://picsum.photos/id/43/640/427.jpg
https://picsum.photos/id/57/640/427.jpg
https://picsum.photos/id/100000/640/427.jpg
"""

In [ ]:
#@title Read the source and preview the filenames
import subprocess

UPLOADS = WORK / "machyon-sources"
UPLOADS.mkdir(exist_ok=True)

if SOURCE == "paste below":
    lines = [u.strip() for u in URLS.splitlines()]
    lines = [u for u in lines if u and not u.startswith("#")]
    if not lines:
        raise ValueError("No links pasted: add one URL per line to URLS above, then run this cell again.")
    SOURCE_PATH = UPLOADS / "pasted.txt"
    SOURCE_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
elif SOURCE == "upload a file":
    if not IN_COLAB:
        raise RuntimeError("Uploading works in Colab. In Jupyter, choose 'file path' and set FILE_PATH.")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file was uploaded.")
    name, data = next(iter(uploaded.items()))
    SOURCE_PATH = UPLOADS / ("upload" + Path(name).suffix.lower())
    SOURCE_PATH.write_bytes(data)
else:
    SOURCE_PATH = Path(FILE_PATH).expanduser()
    if not SOURCE_PATH.is_file():
        raise FileNotFoundError(f"No file at {FILE_PATH!r}. Check FILE_PATH in the settings.")

rows = sum(1 for _ in machyon.read_rows(str(SOURCE_PATH), URL_COLUMN or None))
print(f"{rows} rows in {SOURCE_PATH.name}. The first ones and the files they become:\n")
cmd = [sys.executable, "-m", "machyon", "names", str(SOURCE_PATH), "--limit", "5"]
if URL_COLUMN:
    cmd += ["--url-col", URL_COLUMN]
if FILENAME_TEMPLATE:
    cmd += ["--names", FILENAME_TEMPLATE]
preview = subprocess.run(cmd, capture_output=True, text=True)
print(preview.stdout or preview.stderr)
if preview.returncode:
    raise ValueError("Machyon can't read this source with these settings (see above).")

## 2. Download

Files already in the output folder are skipped, so running this cell again resumes an interrupted download.

In [ ]:
#@title Download
report = await machyon.adownload(str(SOURCE_PATH), OUTPUT_DIR, names=FILENAME_TEMPLATE or None,
                                 url_col=URL_COLUMN or None, **OPTIONS)
print(report)

In [ ]:
#@title Report
from html import escape

from IPython.display import HTML, display

# What to do about a failure, in this notebook's terms (the library's hints name command-line flags).
NOTEBOOK_HINTS = {
    "http_403": "The server refused access. Many sites only serve images to their own pages: set REFERER in "
                "⚙ Settings to a page the images appear on, run that cell, then ⚙ Retry the failures.",
    "not_image": "The server sent a web page instead of an image (an error, login or bot-check page). If the site "
                 "only serves images to its own pages, setting REFERER in ⚙ Settings can help.",
    "timeout": "The server was too slow. Raise TIMEOUT in ⚙ Settings, run that cell, then ⚙ Retry the failures.",
    "http_429": "Rate limited even after backing off. Wait a while, then ⚙ Retry the failures, or lower "
                "MAX_CONCURRENCY in ⚙ Settings.",
    "ssl_certificate": "The server's certificate couldn't be verified, so nothing was downloaded. "
                       "⚙ Check a failing link shows why.",
}


def advice(reason):
    if reason in NOTEBOOK_HINTS:
        return NOTEBOOK_HINTS[reason]
    text = (report.failure_examples.get(reason) or {}).get("hint") or ""
    return text.replace("`machyon retry`", "⚙ Retry the failures")

mb = report.bytes / 1048576
cards = [("Downloaded", report.downloaded), ("Already on disk", report.skipped), ("Failed", report.failed),
         ("Time", f"{report.duration:.1f} s"), ("Speed", f"{report.rate:.1f} img/s"), ("Data", f"{mb:.1f} MB")]
html = "".join(f"<div style='padding:10px 14px;border:1px solid #8884;border-radius:10px'>"
               f"<div style='font-size:12px;opacity:.7'>{k}</div><div style='font-size:20px'>{v}</div></div>"
               for k, v in cards)
html = f"<div style='display:flex;flex-wrap:wrap;gap:8px'>{html}</div>"
if report.failure_reasons:
    items = "".join(f"<li><code>{escape(r)}</code>: {n}. {escape(advice(r))}</li>"
                    for r, n in report.failure_reasons.items())
    html += f"<p><b>Why rows failed</b></p><ul>{items}</ul>"
display(HTML(html))

## 3. Look at the result

In [ ]:
#@title Gallery (up to 24 images)
import base64
import io

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".gif", ".webp", ".bmp", ".tif", ".tiff", ".avif"}
images = sorted(p for p in Path(OUTPUT_DIR).rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def thumbnail(path):
    try:
        from PIL import Image
        with Image.open(path) as im:
            im.thumbnail((240, 240))
            buf = io.BytesIO()
            im.convert("RGB").save(buf, "JPEG", quality=80)
            return "image/jpeg", buf.getvalue()
    except Exception:
        if path.stat().st_size > 2_000_000:
            return None
        return "image/" + path.suffix.lower().lstrip(".").replace("jpg", "jpeg"), path.read_bytes()


tiles = []
for path in images[:24]:
    thumb = thumbnail(path)
    if thumb:
        mime, data = thumb
        name = escape(str(path.relative_to(OUTPUT_DIR)))
        tiles.append(f"<figure style='margin:0;width:120px'><img src='data:{mime};base64,"
                     f"{base64.b64encode(data).decode()}' alt='{name}' style='width:120px;height:90px;"
                     f"object-fit:cover;border-radius:8px'><figcaption style='font-size:11px;overflow:hidden;"
                     f"text-overflow:ellipsis;white-space:nowrap'>{name}</figcaption></figure>")
print(f"{len(images)} images in {OUTPUT_DIR}/" + (f", showing the first {len(tiles)}" if len(images) > 24 else ""))
display(HTML(f"<div style='display:flex;flex-wrap:wrap;gap:8px'>{''.join(tiles)}</div>"))

In [ ]:
#@title Failures
import json

from machyon.ledger import is_permanent

ledger = Path(OUTPUT_DIR) / "failed_downloads.jsonl"
failures = [json.loads(line) for line in ledger.read_text(encoding="utf-8").splitlines() if line.strip()] \
    if ledger.exists() else []

if not failures:
    print("No failures recorded.")
else:
    print(f"{len(failures)} failed link{'s' * (len(failures) != 1)}, recorded in {ledger}. Permanent ones (404, not an image, ...) won't "
          "recover with a retry; the rest may.")
    rows_html = "".join(f"<tr><td><code>{escape(f['reason'])}</code></td><td>{'yes' if is_permanent(f['reason']) else 'no'}"
                        f"</td><td style='word-break:break-all'>{escape(f['url'])}</td>"
                        f"<td>{escape(', '.join(f.get('names', [])))}</td></tr>" for f in failures[:50])
    display(HTML("<table><tr><th>Reason</th><th>Permanent</th><th>URL</th><th>File</th></tr>"
                 f"{rows_html}</table>"))

In [ ]:
#@title ⚙ Check a failing link
#@markdown Runs `machyon doctor` on one link: DNS, proxy, TLS and what the server answers, step by step, with the
#@markdown REFERER from ⚙ Settings. Empty: the first recorded failure that isn't a plain 404.
CHECK_URL = ""  #@param {type:"string"}

url = CHECK_URL.strip() or next((f["url"] for f in failures if f["reason"] != "http_404"),
                                failures[0]["url"] if failures else "")
if not url:
    print("No failures to check. Paste a link into CHECK_URL to check it anyway.")
else:
    cmd = [sys.executable, "-m", "machyon", "doctor", url]
    if REFERER.strip():
        cmd += ["--referer", REFERER.strip()]
    result = subprocess.run(cmd, capture_output=True, text=True)
    text = (result.stdout or result.stderr).replace(   # doctor speaks command-line; say it in this notebook's terms
        "Try --referer with the page the image appears on.",
        "Set REFERER in ⚙ Settings to a page the image appears on, run that cell, then ⚙ Retry the failures.")
    print(text.replace("--referer", "REFERER (⚙ Settings)"))

In [ ]:
#@title ⚙ Retry the failures
#@markdown Retries the recorded failures under their original filenames, with the current ⚙ Settings (change
#@markdown REFERER or TIMEOUT there first if the report suggested it). Tick to include permanent ones too.
RETRY_PERMANENT = False  #@param {type:"boolean"}

async with machyon.Downloader(OUTPUT_DIR, **OPTIONS) as dl:
    retried = await dl.retry_failed(include_permanent=RETRY_PERMANENT)
print(retried if retried else "Nothing to retry" + ("." if RETRY_PERMANENT else
                                                    ": only permanent failures are left. Tick RETRY_PERMANENT to try them anyway."))

## 4. Take the images with you

In [ ]:
#@title ⚙ Save
#@markdown The ZIP holds the output folder, including `failed_downloads.jsonl`. Colab deletes this runtime's files
#@markdown when the session ends, so save before you close the tab.
import shutil

SAVE_TO = "download the ZIP"  #@param ["download the ZIP", "Google Drive", "only make the ZIP"]
DRIVE_FOLDER = "machyon"  #@param {type:"string"}
DRIVE_MOUNT = "/content/drive"

archive = Path(shutil.make_archive(str(WORK / "machyon-images"), "zip", OUTPUT_DIR))
print(f"Made {archive.name} ({archive.stat().st_size / 1048576:.1f} MB)")

if SAVE_TO == "download the ZIP":
    if IN_COLAB:
        from google.colab import files
        files.download(str(archive))
    else:
        from IPython.display import FileLink
        display(FileLink(archive.name))
elif SAVE_TO == "Google Drive":
    if not IN_COLAB:
        raise RuntimeError("Google Drive is available in Colab. In Jupyter, the ZIP is at " + str(archive))
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
    target = Path(DRIVE_MOUNT) / "MyDrive" / DRIVE_FOLDER
    target.mkdir(parents=True, exist_ok=True)
    print("Saved to", shutil.copy2(archive, target / archive.name))

**Next:** the [documentation](https://divyanshusingh96.github.io/machyon/) covers
[resuming large datasets](https://divyanshusingh96.github.io/machyon/reliability/resuming/),
[filename templates](https://divyanshusingh96.github.io/machyon/guides/naming/) and
[every failure reason](https://divyanshusingh96.github.io/machyon/reference/failure-reasons/).
[Machyon in 104 seconds](https://divyanshusingh96.github.io/machyon/project/film/) shows a run of 3,000 links from start
to finish. On your own computer, `machyon ui` gives you the same steps as an app in your browser.
Machyon is MIT-licensed and open source: [github.com/DivyanshuSingh96/machyon](https://github.com/DivyanshuSingh96/machyon).